# 🏐 Volleyball Nations League (VNL) 2026: End-to-End Pipeline & EDA
### Scraping Methodology, Team Imputation & Exploratory Data Analysis

Welcome! This notebook demonstrates the full lifecycle of the **VNL 2026 Player Statistics** dataset:
1. **Data Acquisition Architecture:** How player IDs and detailed metrics were scraped from official Volleyball World pages using Selenium.
2. **Data Cleaning & Standardization:** Handling missing team metadata (`TBD` / `-`).
3. **In-depth Exploratory Data Analysis (EDA):** Performance comparisons across positions, age, height, and national teams.

---
## 1. Scraping Architecture & Data Provenance

The dataset was collected using a two-stage automated pipeline with `undetected-chromedriver`:

1. **Step 1: Harvesting Player IDs (`ids.py`)**  
   Iterated through all individual skill categories (`best-scorers`, `best-attackers`, `best-blockers`, `best-servers`, `best-setters`, `best-diggers`, `best-receivers`) from the official Volleyball World platform to extract unique player profile IDs (`/players/{id}`).

2. **Step 2: Profile Scraping (`script.py`)**  
   Navigated to each player's profile URL, waiting for dynamic DOM rendering, and extracted biometric details (age, height) alongside technical scoring metrics.

3. **Step 3: Team Imputation (`team_fixer.py`)**  
   Players tagged as `TBD` or `-` were mapped to their confirmed national squads using a verified roster dictionary.

---
## 2. Environment Setup & Data Loading

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visual setup
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# Find the file automatically in Kaggle input or current directory
csv_file = None
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('vnl2026.csv'):
            csv_file = os.path.join(root, file)
            break

if not csv_file:
    csv_file = 'vnl2026.csv'

df = pd.read_csv(csv_file)
print(f"Loaded {df.shape[0]} players across {df.shape[1]} features.")
df.head()

---
## 3. Data Cleaning & Type Conversion

Efficiency metrics (`attack_eff`, `block_eff`, `serve_eff`) were stored as strings with `%` symbols, and match averages as text values. We convert them to numerical floats to enable statistical calculations and visualizations.

In [ ]:
# Clean percentage fields
eff_cols = ['attack_eff', 'block_eff', 'serve_eff']
for col in eff_cols:
    df[col] = df[col].astype(str).str.replace('%', '', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Convert match average metrics
avg_cols = ['avg_per_match', 'attack_avg', 'block_avg', 'serve_avg']
for col in avg_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

df[['name', 'team', 'position', 'total_points', 'attack_eff', 'block_eff', 'serve_eff']].info()

---
## 4. Exploratory Data Analysis (EDA)

### 4.1 Player Biometrics: Height and Age Distributions
Let's examine the physical attributes across all tournament competitors.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Height distribution
sns.histplot(data=df, x="height_cm", kde=True, bins=20, ax=axes[0], color="#2b5c8f")
axes[0].set_title("Distribution of Player Height (cm)", weight="bold")
axes[0].set_xlabel("Height (cm)")
axes[0].axvline(df["height_cm"].mean(), color="red", linestyle="--", label=f"Mean: {df['height_cm'].mean():.1f} cm")
axes[0].legend()

# Age distribution
sns.histplot(data=df, x="age", kde=True, bins=15, ax=axes[1], color="#d95f02")
axes[1].set_title("Distribution of Player Age", weight="bold")
axes[1].set_xlabel("Age (years)")
axes[1].axvline(df["age"].mean(), color="red", linestyle="--", label=f"Mean: {df['age'].mean():.1f} yrs")
axes[1].legend()

plt.tight_layout()
plt.show()

### 4.2 Height by Tactical Position
How do physical requirements differ between Liberos, Setters, Middle Blockers, Outside Hitters, and Opposite Spikers?

In [ ]:
plt.figure(figsize=(12, 6))
order = df.groupby("position")["height_cm"].median().sort_values().index
sns.boxplot(data=df, x="position", y="height_cm", order=order, palette="Blues_r")
plt.title("Height Distribution Across Tactical Positions", weight="bold", fontsize=14)
plt.xlabel("Position", weight="bold")
plt.ylabel("Height (cm)", weight="bold")
plt.xticks(rotation=15)
plt.show()

### 4.3 Top 10 Best Scorers
Identifying the tournament's most productive scorers across all participating nations.

In [ ]:
top_scorers = df.sort_values(by="total_points", ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(
    data=top_scorers,
    x="total_points",
    y="name",
    hue="team",
    dodge=False,
    palette="tab10"
)
plt.title("Top 10 Overall Point Scorers", weight="bold", fontsize=14)
plt.xlabel("Total Points Scored", weight="bold")
plt.ylabel("Player", weight="bold")
plt.legend(title="Country", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 4.4 Technical Skill Leaders: Attacks, Blocks, and Serves
A side-by-side comparison of the top 5 specialists in each scoring discipline.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Top Attackers
top_att = df.nlargest(5, "attack_pts")
sns.barplot(data=top_att, x="attack_pts", y="name", ax=axes[0], color="#e41a1c")
axes[0].set_title("Top 5 Attackers (Points)", weight="bold")
axes[0].set_xlabel("Attack Points")

# Top Blockers
top_blk = df.nlargest(5, "block_pts")
sns.barplot(data=top_blk, x="block_pts", y="name", ax=axes[1], color="#377eb8")
axes[1].set_title("Top 5 Blockers (Points)", weight="bold")
axes[1].set_xlabel("Block Points")
# Top Servers
top_srv = df.nlargest(5, "serve_pts")
sns.barplot(data=top_srv, x="serve_pts", y="name", ax=axes[2], color="#4daf4a")
axes[2].set_title("Top 5 Servers (Aces)", weight="bold")
axes[2].set_xlabel("Serve Aces")

plt.tight_layout()
plt.show()

### 4.5 Attack Points vs. Attack Efficiency
Examining scoring efficiency versus volume for key attackers (minimum 30 attack points to remove small-sample noise).

In [ ]:
filtered_att = df[df["attack_pts"] >= 30]

plt.figure(figsize=(11, 7))
sns.scatterplot(
    data=filtered_att,
    x="attack_pts",
    y="attack_eff",
    hue="position",
    size="total_points",
    sizes=(40, 250),
    alpha=0.85
)
plt.title("Attack Points vs. Attack Efficiency (%)", weight="bold", fontsize=14)
plt.xlabel("Total Attack Points", weight="bold")
plt.ylabel("Attack Efficiency (%)", weight="bold")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 4.6 Correlation Matrix
Analyzing linear relationships between player height, age, and skill-specific contributions.

In [ ]:
corr_cols = [
    'age', 'height_cm', 'total_points', 'attack_pts', 
    'attack_eff', 'block_pts', 'block_eff', 'serve_pts', 'serve_eff'
]
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, vmin=-1, vmax=1)
plt.title("Metric Correlation Matrix", weight="bold", fontsize=14)
plt.show()

---
## 5. Summary & Next Steps

### Key Takeaways:
1. **Scraping Pipeline:** `undetected-chromedriver` allowed navigating dynamic Javascript-rendered leaderboards reliably.
2. **Data Consistency:** Missing squad indicators (`TBD` / `-`) were resolved through custom dictionary mapping.
3. **Tactical Insights:** Middle blockers stand out with the highest attack efficiency percentages due to fast-tempo quick attacks, while opposites and outside hitters shoulder the highest offensive volume.

**Potential Extensions:**
* Cluster players with K-Means or UMAP to identify distinct technical profiles.
* Model team offensive strength by aggregating individual player efficiency ratings.